In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import the required libraries
import pandas as pd
import matplotlib.pyplot as plt
import time
import numpy as np
import seaborn as sns
from tabulate import tabulate
from google.colab import files  # remove if the notebook is not run in Colab
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import randint as sp_randint
from scipy.stats import uniform as sp_uniform
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

In [ ]:
# INPUT SELECTION
SEED = 42

MODEL_TAG = 'ctgan_perclass'          # ctgan_perclass | cwgan | tablegan | medgan
DEVICE_TAG = 'gpu'
EPOCHS = 100                     # 100 | 300 | 500
TARGET_PER_CLASS = 6000

STEM = f'{MODEL_TAG}_{DEVICE_TAG}_epoch{EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}'

COMBINED_FILE = f'{STEM}.csv'
SYNTHETIC_ONLY_FILE = f'{STEM}_synthetic_only.csv'

# Which training set this run uses:
TRAINING_FILE = COMBINED_FILE

df_test = pd.read_csv("df_test.csv")
df_train = pd.read_csv(TRAINING_FILE)

print('Training :', TRAINING_FILE, df_train.shape)
print('Test     : df_test.csv', df_test.shape)

## Run configuration, seeds and separation audit

In [ ]:
# RUN CONFIGURATION, SEEDS AND SEPARATION AUDIT
import random, os, platform

SEED = 42
N_SPLITS = 5
SHUFFLE = True
TARGET_COL = 'Target'

random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ---- runtime switches ------------------------------------------
RECORD_CURVE = True

# XGBoost on the GPU. The GAN notebooks already use it; the
USE_GPU = True

XGB_DEVICE_KW = {}
if USE_GPU:
    try:
        import xgboost as _xgb
        if int(_xgb.__version__.split('.')[0]) >= 2:
            XGB_DEVICE_KW = {'device': 'cuda', 'tree_method': 'hist'}
        else:
            XGB_DEVICE_KW = {'tree_method': 'gpu_hist'}
    except Exception as _e:
        print('GPU settings unavailable, falling back to CPU:', _e)

print('=== REPRODUCIBILITY ===')
print('Global seed      :', SEED)
print('Cross-validation : StratifiedKFold(n_splits={}, shuffle={}, random_state={})'
      .format(N_SPLITS, SHUFFLE, SEED))
print('Search           : RandomizedSearchCV(n_iter=5, scoring=f1_macro, random_state={})'
      .format(SEED))
print('Estimators       : random_state={} for LightGBM, XGBoost and Random Forest'
      .format(SEED))
print('Generation seeds : set in notebooks 02-05, same value {}'.format(SEED))
print('Python           :', platform.python_version())
print('Record curve     :', RECORD_CURVE, '| XGBoost GPU:', XGB_DEVICE_KW or 'CPU')

# ---- separation audit -------------------------------------------
_feat = [c for c in df_test.columns if c != TARGET_COL]

_tr = set(map(tuple, np.round(df_train[_feat].to_numpy(dtype=float), 6)))
_te = set(map(tuple, np.round(df_test[_feat].to_numpy(dtype=float), 6)))
_shared = _tr & _te

print('\n=== TRAIN / TEST SEPARATION ===')
print('Training rows          :', len(df_train))
print('Test rows              :', len(df_test))
print('Distinct feature rows shared by the two sets:', len(_shared))

if _shared:
    print('WARNING - the two sets share feature vectors. Since augmentation')
    print('          is applied to the training partition only, a non-zero')
    print('          count means a generator reproduced a held-out record.')
else:
    print('No feature vector appears in both sets. The test partition was')
    print('held out before augmentation and no generator reproduced it.')

print('\nTest class counts :',
      df_test[TARGET_COL].value_counts().sort_index().to_dict())

# ---- tune once, reuse everywhere --------------------------------
import json as _json

TUNE = True
PARAMS_FILE = 'best_params.json'

def _params_store():
    if os.path.exists(PARAMS_FILE):
        with open(PARAMS_FILE) as fh:
            return _json.load(fh)
    return {}

def tuned_params(name, search, X, y):
    """Return the hyperparameters for `name`, searching only if needed."""
    store = _params_store()

    if not TUNE and name in store:
        params = dict(store[name])
        print("Reusing stored hyperparameters for {} from {}:".format(name, PARAMS_FILE))
        for k, v in params.items():
            print("  {}: {}".format(k, v))
        return params

    search.fit(X, y)
    params = dict(search.best_params_)

    print("\n=== Selected hyperparameters ===")
    print("Best Macro F1-Score: {:.4f}".format(search.best_score_))
    for k, v in params.items():
        print("  {}: {}".format(k, v))

    store[name] = params
    with open(PARAMS_FILE, 'w') as fh:
        _json.dump(store, fh, indent=2)
    print("Stored in {}. Set TUNE = False for the remaining training sets."
          .format(PARAMS_FILE))
    return params

print('\nTuning mode      :', 'search and store' if TUNE else 'reuse ' + PARAMS_FILE)

In [ ]:
# Count the label frequencies
train_counts = df_train['Target'].value_counts()
test_counts = df_test['Target'].value_counts()

# Combine both distributions
df_compare = pd.DataFrame({
    'Train': train_counts,
    'Test': test_counts
}).fillna(0)

# Plot
df_compare.plot(kind='bar', figsize=(10,5))
plt.title("Label Distribution Comparison: Train vs Test")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# Lightgbm Classifier


In [ ]:
# === 1. Data preparation ===
TARGET_COL = 'Target'
SEED = 42
N_SPLITS = 5
SHUFFLE = True

X_train = df_train.drop(columns=TARGET_COL)
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=TARGET_COL)
y_test = df_test[TARGET_COL]

# === 2. Hyperparameter search space ===
param_dist = {
    'n_estimators': [100, 150, 200],
    # 0.01 removed: with 50,000+ rows and 13 classes it needs roughly
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 64, 90],
    'max_depth': [3, 5, 10, -1],  # -1 means unlimited depth
    'min_child_samples': [10, 20, 40],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# === 3. Run the randomised search ===
print("=== Running the randomised hyperparameter search ===")
# n_iter=5 evaluates five sampled combinations of the search space.
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=SHUFFLE, random_state=SEED)

base_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    random_state=SEED,
    verbosity=-1
)

random_search = RandomizedSearchCV(
    estimator=base_lgb,
    param_distributions=param_dist,
    n_iter=5, # five sampled combinations
    scoring='f1_macro',
    cv=kf,
    random_state=SEED,
    n_jobs=-1, # Use every available CPU core
    verbose=1
)

best_found = tuned_params('lightgbm', random_search, X_train, y_train)

# === 4. Train and evaluate using the selected hyperparameters ===
best_params = dict(best_found)
best_params.update({'random_state': SEED, 'verbosity': -1, 'objective': 'multiclass'})

# --- Convergence settings for the learning curve -------------
CURVE_MAX_ROUNDS = 2000
CURVE_PATIENCE = 50

best_params['n_estimators'] = CURVE_MAX_ROUNDS
best_model = lgb.LGBMClassifier(**best_params)

best_iterations = []

acc_scores, prec_scores, rec_scores, f1_scores, roc_scores = [], [], [], [], []
all_histories = []

print("\n=== K-fold training and validation with the selected hyperparameters ===")

fold = 1
for train_idx, val_idx in kf.split(X_train, y_train):
    print(f"\n=== Fold {fold} ===")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # The validation set is always supplied, because early stopping
    eval_sets = [(X_tr, y_tr), (X_val, y_val)] if RECORD_CURVE else [(X_val, y_val)]
    eval_names = ['train', 'valid'] if RECORD_CURVE else ['valid']

    model = best_model.fit(
        X_tr, y_tr,
        eval_set=eval_sets,
        eval_names=eval_names,
        eval_metric=['multi_logloss', 'multi_error'],
        callbacks=[lgb.early_stopping(CURVE_PATIENCE, verbose=False),
                   lgb.log_evaluation(0)]
    )

    if RECORD_CURVE:
        all_histories.append(model.evals_result_)
    best_iterations.append(model.best_iteration_ or CURVE_MAX_ROUNDS)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro')
    rec = recall_score(y_val, y_pred, average='macro')
    f1 = f1_score(y_val, y_pred, average='macro')

    y_bin = label_binarize(y_val, classes=np.unique(y_train))
    try:
        roc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    except ValueError:
        roc = np.nan

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)
    f1_scores.append(f1)
    roc_scores.append(roc)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc:.4f}")

    fold += 1

# === Mean results across the folds ===
print("\n=== Mean results of the 5-fold cross-validation ===")
print(f"Mean Accuracy : {np.mean(acc_scores):.4f}")
print(f"Mean Precision: {np.mean(prec_scores):.4f}")
print(f"Mean Recall   : {np.mean(rec_scores):.4f}")
print(f"Mean F1-score : {np.mean(f1_scores):.4f}")
print(f"Mean ROC-AUC  : {np.nanmean(roc_scores):.4f}")

# === Per-fold results ===
results = pd.DataFrame({
    'Fold': range(1, N_SPLITS + 1),
    'Accuracy': acc_scores,
    'Precision': prec_scores,
    'Recall': rec_scores,
    'F1-score': f1_scores,
    'ROC-AUC': roc_scores
})
print("\n=== Per-fold results ===")
print(results.round(4))

# === 5. Final evaluation on the original test set ===
print("\n=== Final evaluation on the original test set ===")

# Final model: the typical number of rounds the folds actually needed
n_final = int(np.median(best_iterations))
print(f"\nRounds used per fold : {best_iterations}")
print(f"Final model rounds   : {n_final}")

if max(best_iterations) >= CURVE_MAX_ROUNDS:
    print("WARNING - a fold reached the ceiling without early stopping.")
    print("          Raise CURVE_MAX_ROUNDS; the curve has not converged.")

final_params = dict(best_params)
final_params['n_estimators'] = n_final
final_model = lgb.LGBMClassifier(**final_params).fit(X_train, y_train)
y_pred_test = final_model.predict(X_test)
y_prob_test = final_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='macro')
test_rec = recall_score(y_test, y_pred_test, average='macro')
test_f1 = f1_score(y_test, y_pred_test, average='macro')

y_bin_test = label_binarize(y_test, classes=np.unique(y_train))
roc_test = roc_auc_score(y_bin_test, y_prob_test, average='macro', multi_class='ovr')

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall   : {test_rec:.4f}")
print(f"Test F1-score : {test_f1:.4f}")
print(f"Test ROC-AUC  : {roc_test:.4f}")

In [ ]:
if not RECORD_CURVE:
    print('RECORD_CURVE is False - no learning curve was recorded for LightGBM.')
else:
    # 1. Align the folds without discarding anything
    def _pad(sequences):
        width = max(len(s) for s in sequences)
        out = np.full((len(sequences), width), np.nan)
        for row, seq in enumerate(sequences):
            out[row, :len(seq)] = seq
        return out

    train_loss_matrix = _pad([h['train']['multi_logloss'] for h in all_histories])
    valid_loss_matrix = _pad([h['valid']['multi_logloss'] for h in all_histories])
    train_acc_matrix = 1.0 - _pad([h['train']['multi_error'] for h in all_histories])
    valid_acc_matrix = 1.0 - _pad([h['valid']['multi_error'] for h in all_histories])

    # 2. Mean and standard deviation across folds
    mean_train_loss, std_train_loss = np.nanmean(train_loss_matrix, axis=0), np.nanstd(train_loss_matrix, axis=0)
    mean_valid_loss, std_valid_loss = np.nanmean(valid_loss_matrix, axis=0), np.nanstd(valid_loss_matrix, axis=0)

    mean_train_acc, std_train_acc = np.nanmean(train_acc_matrix, axis=0), np.nanstd(train_acc_matrix, axis=0)
    mean_valid_acc, std_valid_acc = np.nanmean(valid_acc_matrix, axis=0), np.nanstd(valid_acc_matrix, axis=0)

    # 3. Learning curves
    plt.figure(figsize=(12, 4.5))

    # Plot Log Loss
    plt.subplot(1, 2, 1)
    epochs_loss = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_loss, mean_train_loss, label='Train Loss (Mean)', color='blue')
    plt.fill_between(epochs_loss, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, color='blue', alpha=0.15)

    plt.plot(epochs_loss, mean_valid_loss, label='Valid Loss (Mean)', color='orange')
    plt.fill_between(epochs_loss, mean_valid_loss - std_valid_loss, mean_valid_loss + std_valid_loss, color='orange', alpha=0.15)

    plt.title('Mean Log Loss Across 5 Folds')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    epochs_acc = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_acc, mean_train_acc, label='Train Acc (Mean)', color='blue')
    plt.fill_between(epochs_acc, mean_train_acc - std_train_acc, mean_train_acc + std_train_acc, color='blue', alpha=0.15)

    plt.plot(epochs_acc, mean_valid_acc, label='Valid Acc (Mean)', color='orange')
    plt.fill_between(epochs_acc, mean_valid_acc - std_valid_acc, mean_valid_acc + std_valid_acc, color='orange', alpha=0.15)

    plt.title('Mean Accuracy Across 5 Folds')
    plt.xlabel('Iterations')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('learning_curve_lightgbm.png', dpi=300) # High-resolution figure
    plt.show()

    # 4. Has the curve converged?
    tail = max(5, valid_loss_matrix.shape[1] // 10)
    recent = np.nanmean(mean_valid_loss[-tail:])
    earlier = np.nanmean(mean_valid_loss[-2 * tail:-tail])
    drift = (earlier - recent) / max(abs(earlier), 1e-12)

    print('Iterations plotted       :', valid_loss_matrix.shape[1])
    print('Validation loss, last 10% : {:.6f}'.format(recent))
    print('Validation loss, prior 10%: {:.6f}'.format(earlier))
    print('Relative improvement     : {:.4%}'.format(drift))

    if drift > 0.001:
        print('\nNOT CONVERGED - the curve is still improving.')
        print('Raise the ceiling and re-run before using this figure.')
    elif recent > earlier:
        print('\nValidation loss is rising: past the optimum, early stopping was right.')
    else:
        print('\nCONVERGED - the curve has flattened. Safe to use in the paper.')

In [ ]:
# Basic metrics
acc  = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, average='macro')
rec  = recall_score(y_test, y_pred_test, average='macro')
f1   = f1_score(y_test, y_pred_test, average='macro')

print("\nPerformance Metrics:")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

# Multi-class ROC-AUC
try:
    if len(np.unique(y_test)) < 2:
        print("ROC-AUC cannot be computed: y_test contains a single class.")
        auc = np.nan
    else:
        y_proba = final_model.predict_proba(X_test)  # use the fitted final model
        y_bin_test = label_binarize(y_test, classes=np.unique(y_train))
        auc = roc_auc_score(y_bin_test, y_proba, multi_class='ovr', average='macro')
        print(f"ROC-AUC   : {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC   : could not be computed ({e})")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
labels = sorted(y_test.unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)

plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Sensitivity and specificity (multi-class definition)
sensitivity = recall_score(y_test, y_pred_test, average='macro')

specificity_list = []
for i in range(len(cm)):
    tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
    fp = np.sum(cm[:, i]) - cm[i, i]
    specificity_list.append(tn / (tn + fp))
specificity = np.mean(specificity_list)

print(f"Sensitivity (Macro Recall): {sensitivity:.4f}")
print(f"Specificity (Macro)       : {specificity:.4f}")

# VERIFICATION - reviewer point on result discrepancies
cm_accuracy = np.trace(cm) / cm.sum()

print("\n=== ACCURACY CROSS-CHECK ===")
print("accuracy_score        : {:.6f}".format(acc))
print("trace(cm) / sum(cm)   : {:.6f}".format(cm_accuracy))
print("difference            : {:.2e}".format(abs(acc - cm_accuracy)))
print("correct / total       : {:,} / {:,}".format(int(np.trace(cm)), int(cm.sum())))

assert abs(acc - cm_accuracy) < 1e-9, \
    "Confusion matrix and accuracy_score disagree - do not report these together."

# ---- per-class counts, exactly as they appear in the matrix -----
per_class = pd.DataFrame({
    'Class': labels,
    'Support': [int(cm[i].sum()) for i in range(len(labels))],
    'Correct': [int(cm[i, i]) for i in range(len(labels))],
    'Recall': [cm[i, i] / cm[i].sum() for i in range(len(labels))],
})
per_class['Errors'] = per_class['Support'] - per_class['Correct']

print("\n=== PER-CLASS COUNTS FROM THE CONFUSION MATRIX ===")
print(per_class.to_string(index=False,
                          formatters={'Recall': '{:.4f}'.format}))

perfect = per_class[per_class['Errors'] == 0]['Class'].tolist()
if perfect:
    print("\nPerfectly classified classes:", perfect)
    print("These are reported as 1000/1000 in the paper. The relevant")
    print("check is the separation audit at the top of this notebook:")
    print("the test partition was held out before augmentation and")
    print("shares no feature vector with the training set, so a perfect")
    print("score here reflects separability in the source data, not")
    print("leakage. Notebook 09 confirms it by reproducing the same")
    print("result from the un-augmented baseline.")

# XGBoost Classifier


In [ ]:
# Convergence settings, shared with the LightGBM cell above
CURVE_MAX_ROUNDS = 2000
CURVE_PATIENCE = 50

# === 1. Data preparation ===
TARGET_COL = 'Target'
SEED = 42
N_SPLITS = 5
SHUFFLE = True

X_train = df_train.drop(columns=TARGET_COL)
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=TARGET_COL)
y_test = df_test[TARGET_COL]

# === 2. Hyperparameter search space ===
param_dist = {
    'n_estimators': [100, 150, 200],
    # 0.01 removed: with 50,000+ rows and 13 classes it needs roughly
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 10, 0],  # 0 means unlimited depth
    'min_child_weight': [1, 3, 5], # Equivalent to min_child_samples
    'gamma': [0, 0.1, 0.2], # Additional parameter that limits overfitting
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# === 3. Run the randomised search ===
print("=== Running the randomised hyperparameter search ===")
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=SHUFFLE, random_state=SEED)

base_xgb = xgb.XGBClassifier(
    objective='multi:softprob',
    random_state=SEED,
    verbosity=0,
    eval_metric='mlogloss'
)

random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=5,
    scoring='f1_macro',
    cv=kf,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

best_found = tuned_params('xgboost', random_search, X_train, y_train)

# === 4. Train and evaluate using the selected hyperparameters ===
best_params = dict(best_found)
best_params.update({
    'random_state': SEED,
    'verbosity': 0,
    'objective': 'multi:softprob',
    'eval_metric': ['mlogloss', 'merror'],
    'early_stopping_rounds': CURVE_PATIENCE,
    'n_estimators': CURVE_MAX_ROUNDS,
    **XGB_DEVICE_KW,
})

best_model = xgb.XGBClassifier(**best_params)

acc_scores, prec_scores, rec_scores, f1_scores, roc_scores = [], [], [], [], []
all_histories = []
best_iterations = []

print("\n=== K-fold training and validation with the selected hyperparameters ===")

fold = 1
for train_idx, val_idx in kf.split(X_train, y_train):
    print(f"\n=== Fold {fold} ===")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # The validation set is always supplied, because early stopping
    eval_sets = [(X_tr, y_tr), (X_val, y_val)] if RECORD_CURVE else [(X_val, y_val)]

    model = best_model.fit(
        X_tr, y_tr,
        eval_set=eval_sets,
        verbose=False
    )

    if RECORD_CURVE:
        all_histories.append(model.evals_result())

    best_iterations.append(
        getattr(model, 'best_iteration', None) or CURVE_MAX_ROUNDS)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro')
    rec = recall_score(y_val, y_pred, average='macro')
    f1 = f1_score(y_val, y_pred, average='macro')

    y_bin = label_binarize(y_val, classes=np.unique(y_train))
    try:
        roc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    except ValueError:
        roc = np.nan

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)
    f1_scores.append(f1)
    roc_scores.append(roc)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc:.4f}")

    fold += 1

# === Mean results across the folds ===
print("\n=== Mean results of the 5-fold cross-validation ===")
print(f"Mean Accuracy : {np.mean(acc_scores):.4f}")
print(f"Mean Precision: {np.mean(prec_scores):.4f}")
print(f"Mean Recall   : {np.mean(rec_scores):.4f}")
print(f"Mean F1-score : {np.mean(f1_scores):.4f}")
print(f"Mean ROC-AUC  : {np.nanmean(roc_scores):.4f}")

# === Per-fold results ===
results = pd.DataFrame({
    'Fold': range(1, N_SPLITS + 1),
    'Accuracy': acc_scores,
    'Precision': prec_scores,
    'Recall': rec_scores,
    'F1-score': f1_scores,
    'ROC-AUC': roc_scores
})
print("\n=== Per-fold results ===")
print(results.round(4))

# === 5. Final evaluation on the original test set ===
print("\n=== Final evaluation on the original test set ===")

# Copy the selected parameters and drop early_stopping_rounds for the final fit
final_params = dict(best_params)
final_params.pop('early_stopping_rounds', None)
final_params['n_estimators'] = int(np.median(best_iterations))

print("\nRounds used per fold :", best_iterations)
print("Final model rounds   :", final_params['n_estimators'])

if max(best_iterations) >= CURVE_MAX_ROUNDS:
    print("WARNING - a fold reached the ceiling without early stopping.")
    print("          Raise CURVE_MAX_ROUNDS; the curve has not converged.")

final_model = xgb.XGBClassifier(**final_params)
final_model.fit(X_train, y_train, verbose=False)

y_pred_test = final_model.predict(X_test)
y_prob_test = final_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='macro')
test_rec = recall_score(y_test, y_pred_test, average='macro')
test_f1 = f1_score(y_test, y_pred_test, average='macro')

y_bin_test = label_binarize(y_test, classes=np.unique(y_train))
roc_test = roc_auc_score(y_bin_test, y_prob_test, average='macro', multi_class='ovr')

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall   : {test_rec:.4f}")
print(f"Test F1-score : {test_f1:.4f}")
print(f"Test ROC-AUC  : {roc_test:.4f}")

In [ ]:
if not RECORD_CURVE:
    print('RECORD_CURVE is False - no learning curve was recorded for XGBoost.')
else:
    # 1. Align the folds without discarding anything
    def _pad(sequences):
        width = max(len(s) for s in sequences)
        out = np.full((len(sequences), width), np.nan)
        for row, seq in enumerate(sequences):
            out[row, :len(seq)] = seq
        return out

    train_loss_matrix = _pad([h['validation_0']['mlogloss'] for h in all_histories])
    valid_loss_matrix = _pad([h['validation_1']['mlogloss'] for h in all_histories])
    train_acc_matrix = 1.0 - _pad([h['validation_0']['merror'] for h in all_histories])
    valid_acc_matrix = 1.0 - _pad([h['validation_1']['merror'] for h in all_histories])

    # 2. Mean and standard deviation across folds
    mean_train_loss, std_train_loss = np.nanmean(train_loss_matrix, axis=0), np.nanstd(train_loss_matrix, axis=0)
    mean_valid_loss, std_valid_loss = np.nanmean(valid_loss_matrix, axis=0), np.nanstd(valid_loss_matrix, axis=0)

    mean_train_acc, std_train_acc = np.nanmean(train_acc_matrix, axis=0), np.nanstd(train_acc_matrix, axis=0)
    mean_valid_acc, std_valid_acc = np.nanmean(valid_acc_matrix, axis=0), np.nanstd(valid_acc_matrix, axis=0)

    # 3. Learning curves
    plt.figure(figsize=(12, 4.5))

    # Plot Log Loss
    plt.subplot(1, 2, 1)
    epochs_loss = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_loss, mean_train_loss, label='Train Loss (Mean)', color='blue')
    plt.fill_between(epochs_loss, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, color='blue', alpha=0.15)

    plt.plot(epochs_loss, mean_valid_loss, label='Valid Loss (Mean)', color='orange')
    plt.fill_between(epochs_loss, mean_valid_loss - std_valid_loss, mean_valid_loss + std_valid_loss, color='orange', alpha=0.15)

    plt.title('Mean Log Loss Across 5 Folds')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    epochs_acc = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_acc, mean_train_acc, label='Train Acc (Mean)', color='blue')
    plt.fill_between(epochs_acc, mean_train_acc - std_train_acc, mean_train_acc + std_train_acc, color='blue', alpha=0.15)

    plt.plot(epochs_acc, mean_valid_acc, label='Valid Acc (Mean)', color='orange')
    plt.fill_between(epochs_acc, mean_valid_acc - std_valid_acc, mean_valid_acc + std_valid_acc, color='orange', alpha=0.15)

    plt.title('Mean Accuracy Across 5 Folds')
    plt.xlabel('Iterations')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('learning_curve_xgboost.png', dpi=300)
    plt.show()

    # 4. Has the curve converged?
    tail = max(5, valid_loss_matrix.shape[1] // 10)
    recent = np.nanmean(mean_valid_loss[-tail:])
    earlier = np.nanmean(mean_valid_loss[-2 * tail:-tail])
    drift = (earlier - recent) / max(abs(earlier), 1e-12)

    print('Iterations plotted       :', valid_loss_matrix.shape[1])
    print('Validation loss, last 10% : {:.6f}'.format(recent))
    print('Validation loss, prior 10%: {:.6f}'.format(earlier))
    print('Relative improvement     : {:.4%}'.format(drift))

    if drift > 0.001:
        print('\nNOT CONVERGED - the curve is still improving.')
        print('Raise the ceiling and re-run before using this figure.')
    elif recent > earlier:
        print('\nValidation loss is rising: past the optimum, early stopping was right.')
    else:
        print('\nCONVERGED - the curve has flattened. Safe to use in the paper.')

In [ ]:
# Basic metrics
acc  = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, average='macro')
rec  = recall_score(y_test, y_pred_test, average='macro')
f1   = f1_score(y_test, y_pred_test, average='macro')

print("\nPerformance Metrics:")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

# Multi-class ROC-AUC
try:
    if len(np.unique(y_test)) < 2:
        print("ROC-AUC cannot be computed: y_test contains a single class.")
        auc = np.nan
    else:
        y_proba = final_model.predict_proba(X_test)
        y_bin_test = label_binarize(y_test, classes=np.unique(y_train))
        auc = roc_auc_score(y_bin_test, y_proba, multi_class='ovr', average='macro')
        print(f"ROC-AUC   : {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC   : could not be computed ({e})")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
labels = sorted(y_test.unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)

plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Sensitivity (macro recall)
sensitivity = recall_score(y_test, y_pred_test, average='macro')
print(f"Sensitivity (Macro Recall): {sensitivity:.4f}")

# VERIFICATION - reviewer point on result discrepancies
cm_accuracy = np.trace(cm) / cm.sum()

print("\n=== ACCURACY CROSS-CHECK ===")
print("accuracy_score        : {:.6f}".format(acc))
print("trace(cm) / sum(cm)   : {:.6f}".format(cm_accuracy))
print("difference            : {:.2e}".format(abs(acc - cm_accuracy)))
print("correct / total       : {:,} / {:,}".format(int(np.trace(cm)), int(cm.sum())))

assert abs(acc - cm_accuracy) < 1e-9, \
    "Confusion matrix and accuracy_score disagree - do not report these together."

# ---- per-class counts, exactly as they appear in the matrix -----
per_class = pd.DataFrame({
    'Class': labels,
    'Support': [int(cm[i].sum()) for i in range(len(labels))],
    'Correct': [int(cm[i, i]) for i in range(len(labels))],
    'Recall': [cm[i, i] / cm[i].sum() for i in range(len(labels))],
})
per_class['Errors'] = per_class['Support'] - per_class['Correct']

print("\n=== PER-CLASS COUNTS FROM THE CONFUSION MATRIX ===")
print(per_class.to_string(index=False,
                          formatters={'Recall': '{:.4f}'.format}))

perfect = per_class[per_class['Errors'] == 0]['Class'].tolist()
if perfect:
    print("\nPerfectly classified classes:", perfect)
    print("These are reported as 1000/1000 in the paper. The relevant")
    print("check is the separation audit at the top of this notebook:")
    print("the test partition was held out before augmentation and")
    print("shares no feature vector with the training set, so a perfect")
    print("score here reflects separability in the source data, not")
    print("leakage. Notebook 09 confirms it by reproducing the same")
    print("result from the un-augmented baseline.")

# Random Forest Classifier


In [ ]:
# === 1. Data preparation ===
TARGET_COL = 'Target'
SEED = 42
N_SPLITS = 5
SHUFFLE = True

X_train = df_train.drop(columns=TARGET_COL)
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=TARGET_COL)
y_test = df_test[TARGET_COL]

# === 2. Hyperparameter search space ===
param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# === 3. Run the randomised search ===
print("=== Running the randomised hyperparameter search ===")
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=SHUFFLE, random_state=SEED)

base_rf = RandomForestClassifier(
    random_state=SEED,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=5,
    scoring='f1_macro',
    cv=kf,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

best_found = tuned_params('random_forest', random_search, X_train, y_train)

# === 4. Train and evaluate using the selected hyperparameters ===
best_params = dict(best_found)
best_params.update({'random_state': SEED, 'n_jobs': -1})

acc_scores, prec_scores, rec_scores, f1_scores, roc_scores = [], [], [], [], []
all_histories = []

print("\n=== K-fold training and validation with the selected hyperparameters ===")

fold = 1
classes_list = np.unique(y_train)

for train_idx, val_idx in kf.split(X_train, y_train):
    print(f"\n=== Fold {fold} ===")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # warm_start=True allows the learning curve to be recorded as trees are added
    rf_fold = RandomForestClassifier(**best_params, warm_start=True)

    # A Random Forest does not overfit as trees are added, so the
    CURVE_MAX_TREES = 300

    if RECORD_CURVE:
        target_trees = CURVE_MAX_TREES
        history = {
            'train': {'multi_logloss': [], 'multi_error': []},
            'valid': {'multi_logloss': [], 'multi_error': []}
        }

        step = 5
        for n_trees in range(step, target_trees + 1, step):
            rf_fold.n_estimators = n_trees
            rf_fold.fit(X_tr, y_tr)

            tr_prob = rf_fold.predict_proba(X_tr)
            tr_pred = rf_fold.predict(X_tr)
            history['train']['multi_logloss'].append(
                log_loss(y_tr, tr_prob, labels=classes_list))
            history['train']['multi_error'].append(
                1.0 - accuracy_score(y_tr, tr_pred))

            val_prob = rf_fold.predict_proba(X_val)
            val_pred = rf_fold.predict(X_val)
            history['valid']['multi_logloss'].append(
                log_loss(y_val, val_prob, labels=classes_list))
            history['valid']['multi_error'].append(
                1.0 - accuracy_score(y_val, val_pred))

        all_histories.append(history)
    else:
        rf_fold.n_estimators = best_params.get('n_estimators', 100)
        rf_fold.fit(X_tr, y_tr)

    y_pred = rf_fold.predict(X_val)
    y_prob = rf_fold.predict_proba(X_val)

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro')
    rec = recall_score(y_val, y_pred, average='macro')
    f1 = f1_score(y_val, y_pred, average='macro')

    y_bin = label_binarize(y_val, classes=classes_list)
    try:
        roc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    except ValueError:
        roc = np.nan

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)
    f1_scores.append(f1)
    roc_scores.append(roc)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc:.4f}")

    fold += 1

# === Mean results across the folds ===
print("\n=== Mean results of the 5-fold cross-validation ===")
print(f"Mean Accuracy : {np.mean(acc_scores):.4f}")
print(f"Mean Precision: {np.mean(prec_scores):.4f}")
print(f"Mean Recall   : {np.mean(rec_scores):.4f}")
print(f"Mean F1-score : {np.mean(f1_scores):.4f}")
print(f"Mean ROC-AUC  : {np.nanmean(roc_scores):.4f}")

# === Per-fold results ===
results = pd.DataFrame({
    'Fold': range(1, N_SPLITS + 1),
    'Accuracy': acc_scores,
    'Precision': prec_scores,
    'Recall': rec_scores,
    'F1-score': f1_scores,
    'ROC-AUC': roc_scores
})
print("\n=== Per-fold results ===")
print(results.round(4))

# === 5. Final evaluation on the original test set ===
print("\n=== Final evaluation on the original test set ===")

final_model = RandomForestClassifier(**best_params)
final_model.fit(X_train, y_train)

y_pred_test = final_model.predict(X_test)
y_prob_test = final_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='macro')
test_rec = recall_score(y_test, y_pred_test, average='macro')
test_f1 = f1_score(y_test, y_pred_test, average='macro')

y_bin_test = label_binarize(y_test, classes=classes_list)
roc_test = roc_auc_score(y_bin_test, y_prob_test, average='macro', multi_class='ovr')

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall   : {test_rec:.4f}")
print(f"Test F1-score : {test_f1:.4f}")
print(f"Test ROC-AUC  : {roc_test:.4f}")

In [ ]:
if not RECORD_CURVE:
    print('RECORD_CURVE is False - no learning curve was recorded for Random Forest.')
else:
    # 1. Align the folds without discarding anything
    def _pad(sequences):
        width = max(len(s) for s in sequences)
        out = np.full((len(sequences), width), np.nan)
        for row, seq in enumerate(sequences):
            out[row, :len(seq)] = seq
        return out

    train_loss_matrix = _pad([h['train']['multi_logloss'] for h in all_histories])
    valid_loss_matrix = _pad([h['valid']['multi_logloss'] for h in all_histories])
    train_acc_matrix = 1.0 - _pad([h['train']['multi_error'] for h in all_histories])
    valid_acc_matrix = 1.0 - _pad([h['valid']['multi_error'] for h in all_histories])

    # 2. Mean and standard deviation across folds
    mean_train_loss, std_train_loss = np.nanmean(train_loss_matrix, axis=0), np.nanstd(train_loss_matrix, axis=0)
    mean_valid_loss, std_valid_loss = np.nanmean(valid_loss_matrix, axis=0), np.nanstd(valid_loss_matrix, axis=0)

    mean_train_acc, std_train_acc = np.nanmean(train_acc_matrix, axis=0), np.nanstd(train_acc_matrix, axis=0)
    mean_valid_acc, std_valid_acc = np.nanmean(valid_acc_matrix, axis=0), np.nanstd(valid_acc_matrix, axis=0)

    # 3. Learning curves
    plt.figure(figsize=(12, 4.5))

    # Plot Log Loss
    plt.subplot(1, 2, 1)
    epochs_loss = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_loss, mean_train_loss, label='Train Loss (Mean)', color='blue')
    plt.fill_between(epochs_loss, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, color='blue', alpha=0.15)

    plt.plot(epochs_loss, mean_valid_loss, label='Valid Loss (Mean)', color='orange')
    plt.fill_between(epochs_loss, mean_valid_loss - std_valid_loss, mean_valid_loss + std_valid_loss, color='orange', alpha=0.15)

    plt.title('Mean Log Loss Across 5 Folds')
    plt.xlabel('Trees / Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    epochs_acc = range(valid_loss_matrix.shape[1])
    plt.plot(epochs_acc, mean_train_acc, label='Train Acc (Mean)', color='blue')
    plt.fill_between(epochs_acc, mean_train_acc - std_train_acc, mean_train_acc + std_train_acc, color='blue', alpha=0.15)

    plt.plot(epochs_acc, mean_valid_acc, label='Valid Acc (Mean)', color='orange')
    plt.fill_between(epochs_acc, mean_valid_acc - std_valid_acc, mean_valid_acc + std_valid_acc, color='orange', alpha=0.15)

    plt.title('Mean Accuracy Across 5 Folds')
    plt.xlabel('Trees / Iterations')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('learning_curve_random_forest.png', dpi=300)
    plt.show()

    # 4. Has the curve converged?
    tail = max(5, valid_loss_matrix.shape[1] // 10)
    recent = np.nanmean(mean_valid_loss[-tail:])
    earlier = np.nanmean(mean_valid_loss[-2 * tail:-tail])
    drift = (earlier - recent) / max(abs(earlier), 1e-12)

    print('Iterations plotted       :', valid_loss_matrix.shape[1])
    print('Validation loss, last 10% : {:.6f}'.format(recent))
    print('Validation loss, prior 10%: {:.6f}'.format(earlier))
    print('Relative improvement     : {:.4%}'.format(drift))

    if drift > 0.001:
        print('\nNOT CONVERGED - the curve is still improving.')
        print('Raise the ceiling and re-run before using this figure.')
    elif recent > earlier:
        print('\nValidation loss is rising: past the optimum, early stopping was right.')
    else:
        print('\nCONVERGED - the curve has flattened. Safe to use in the paper.')

In [ ]:
# Basic metrics
acc  = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, average='macro')
rec  = recall_score(y_test, y_pred_test, average='macro')
f1   = f1_score(y_test, y_pred_test, average='macro')

print("\nPerformance Metrics:")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

# Multi-class ROC-AUC
try:
    if len(np.unique(y_test)) < 2:
        print("ROC-AUC cannot be computed: y_test contains a single class.")
        auc = np.nan
    else:
        y_proba = final_model.predict_proba(X_test)
        y_bin_test = label_binarize(y_test, classes=np.unique(y_train))
        auc = roc_auc_score(y_bin_test, y_proba, multi_class='ovr', average='macro')
        print(f"ROC-AUC   : {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC   : could not be computed ({e})")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
labels = sorted(y_test.unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)

plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Sensitivity (macro recall)
sensitivity = recall_score(y_test, y_pred_test, average='macro')
print(f"Sensitivity (Macro Recall): {sensitivity:.4f}")

# VERIFICATION - reviewer point on result discrepancies
cm_accuracy = np.trace(cm) / cm.sum()

print("\n=== ACCURACY CROSS-CHECK ===")
print("accuracy_score        : {:.6f}".format(acc))
print("trace(cm) / sum(cm)   : {:.6f}".format(cm_accuracy))
print("difference            : {:.2e}".format(abs(acc - cm_accuracy)))
print("correct / total       : {:,} / {:,}".format(int(np.trace(cm)), int(cm.sum())))

assert abs(acc - cm_accuracy) < 1e-9, \
    "Confusion matrix and accuracy_score disagree - do not report these together."

# ---- per-class counts, exactly as they appear in the matrix -----
per_class = pd.DataFrame({
    'Class': labels,
    'Support': [int(cm[i].sum()) for i in range(len(labels))],
    'Correct': [int(cm[i, i]) for i in range(len(labels))],
    'Recall': [cm[i, i] / cm[i].sum() for i in range(len(labels))],
})
per_class['Errors'] = per_class['Support'] - per_class['Correct']

print("\n=== PER-CLASS COUNTS FROM THE CONFUSION MATRIX ===")
print(per_class.to_string(index=False,
                          formatters={'Recall': '{:.4f}'.format}))

perfect = per_class[per_class['Errors'] == 0]['Class'].tolist()
if perfect:
    print("\nPerfectly classified classes:", perfect)
    print("These are reported as 1000/1000 in the paper. The relevant")
    print("check is the separation audit at the top of this notebook:")
    print("the test partition was held out before augmentation and")
    print("shares no feature vector with the training set, so a perfect")
    print("score here reflects separability in the source data, not")
    print("leakage. Notebook 09 confirms it by reproducing the same")
    print("result from the un-augmented baseline.")